<p align="center">
  <img src="./yibo quant.jpg" alt="课程封面" width="1300"/>
</p>


# 《和Yibo零基础学习量化金融》
## 从 Python 到 AI 量化交易实战 · Phase 2：风险与组合管理
### 第五章：理解波动率

---

## 本章你将学会

- ✅ 为什么 **收益相同，投资体验却可以天差地别**
- ✅ 什么是 **波动率**，以及它衡量的到底是什么
- ✅ 如何用代码 **计算并比较** 不同股票的波动率
- ✅ 为什么量化研究员 **不仅看收益，更关注风险**

**本章难度**  
⭐⭐☆☆☆

**预计学习时间**  
30～45 分钟（需联网下载行情）

**前置知识**

- 第一期第二章：收益率、`pct_change()`
- Pandas、Matplotlib 基础
- 会用 Jupyter 从上到下运行单元格

---

Yibo 先说句心里话：

很多教程会把波动率讲成「标准差，完事了」。但你在真实账户里感受到的 **焦虑、失眠、想割肉**，才是这一章真正要对接的东西。

本章教学顺序：

```
现象 → 问题 → 直觉 → 代码 → 可视化 → 公式 → 验证 → 思考
```

准备好了吗？我们从一个小故事开始。

# 波动率理论发展脉络

## 1. 数学源头：Louis Bachelier 路易·巴舍利耶（1900）【见第一期内容】
金融领域**最早引入随机波动数学框架**的学者。在博士论文《投机理论》中，首次利用布朗运动描述资产价格波动，是**波动率数学建模的起点**。

- **贡献**：首次使用标准差量化资产价格震荡，构建算术布朗运动，完成金融波动最早的统计量化定义。
- **局限**：假设价格服从正态随机游走，未区分价格与收益率，同时忽略资产价格不能为负的现实约束。

<p align="center">
  <img src="./Harry Markowitz.jpg" alt="人物图片" width="600"/>
</p>

## 2. 标准化波动率奠基：Harry Markowitz 哈里·马科维茨（1952）
在经典论文《资产组合选择》中，**第一次将方差/波动率正式定义为金融风险的量化指标**，建立严谨数理体系，使波动率成为可计算、可优化的标准风险度量。

> ✨ **核心里程碑**
> 在马科维茨之前，市场只有“波动大=风险高”的主观直觉；
> 他首次将波动率**标准化、数学化、可量化**，奠定现代资产组合理论基础，于1990年获得诺贝尔经济学奖。


<p align="center">
  <img src="./Benoit Mandelbrot.jpg" alt="人物图片" width="400"/>
</p>

## 3. 全球第一次波动率实证实验：Benoit Mandelbrot 本华·曼德博（1963）
人类金融史上**首个系统性、大规模波动率实证研究**。
曼德博在论文《某些投机价格的变动》中，利用**百年棉花期货日度数据**完成波动率规律实证，彻底颠覆传统正态分布假设。

**实验三大颠覆性结论：**
- 金融收益率**不服从正态分布**，存在显著厚尾、极端风险；
- 发现**波动率聚集效应**：高波动、低波动会成簇出现，波动具有时序相关性；
- 证明**常数波动率假设完全不符合真实市场**，为后续时变波动率模型奠定实证基础。

<p align="center">
  <img src="./Robert Engle.jpg" alt="人物图片" width="400"/>
</p>

## 4. 时变波动率建模奠基人：Robert Engle 罗伯特·恩格尔（1982）
提出 **ARCH（自回归条件异方差）模型**，正式建立**时变波动率理论**，完美量化解释曼德博发现的波动率聚集现象。
该成果使恩格尔获得 **2003 年诺贝尔经济学奖**。

- **核心突破**：放弃传统“波动率恒定”假设，允许波动率随历史收益动态演化，成为现代波动率预测、期权定价、波动率交易的核心工具。
- **重要延伸**：1986 年 Bollerslev 提出 **GARCH(1,1)**，成为业界最通用的波动率基准模型。

### 环境准备

老规矩，先跑这一格。缺库就在项目根目录执行：`pip install -r requirements.txt`

In [ ]:
# ========== 环境准备 ==========
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['font.sans-serif'] = ['SimHei']      # Windows；Mac 可改 'PingFang SC'
plt.rcParams['axes.unicode_minus'] = False

TRADING_DAYS = 252   # 美股一年大约 252 个交易日


def get_close(data, ticker=None):
    """兼容 yfinance 单标的 / 多标的返回格式"""
    close = data['Close']
    if isinstance(close, pd.DataFrame):
        return close[ticker] if ticker else close.squeeze()
    return close

print('环境就绪 ✓')

---

### 5.1 为什么两只收益相同的股票，体验完全不同？

想象你和朋友各拿 **1 万元**，分别买入两只股票，连续观察 **5 天**：

**股票 A**

```
100 → 105 → 110 → 115 → 120
```

**股票 B**

```
100 → 80 → 130 → 70 → 120
```

第 5 天，两个人账户都是 **12,000 元**——相对本金都赚了 **20%**。

先别急着看代码。停下来想 10 秒：

> **你更愿意持有哪一只？为什么？**

大多数人的直觉是 A。因为 B 中间经历过：

- 从 100 跌到 80（**-20%**）
- 从 130 跌到 70（**-46%** 级别的大回撤）

**收益一样，体验却完全不同。**

这就是本章的核心问题：

> **量化研究员不仅看「赚多少」，还要看「中间有多颠」。**

**目的**：把上面的故事画成图，并算一算两只股票的总涨幅是否真的相同。

In [ ]:
# ========== 5.1 两种路径，同一终点 ==========
days = ['Day 1', 'Day 2', 'Day 3', 'Day 4', 'Day 5']
x = np.arange(len(days))
stock_a = [100, 105, 110, 115, 120]
stock_b = [100, 80, 130, 70, 120]

path_table = pd.DataFrame({'股票A': stock_a, '股票B': stock_b}, index=days)
print('价格路径对照：')
display(path_table)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, stock_a, 'o-', color='#007AFF', linewidth=2.5, markersize=9, label='股票 A · 平稳上涨')
ax.plot(x, stock_b, 's-', color='#E82127', linewidth=2.5, markersize=9, label='股票 B · 大起大落')
ax.axhline(120, color='gray', linestyle='--', alpha=0.5, label='终点都是 120')
ax.set_xticks(x)
ax.set_xticklabels(days)
ax.set_ylabel('价格')
ax.set_title('收益相同，路径不同：你更愿意持有哪一只？', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ret_a = stock_a[-1] / stock_a[0] - 1
ret_b = stock_b[-1] / stock_b[0] - 1
print(f'股票 A 总涨幅: {ret_a:.0%}')
print(f'股票 B 总涨幅: {ret_b:.0%}')

**结果怎么读？**

- 两只股票 **最终收益完全相同**（都是 +20%）。
- 但 B 的路径像 **过山车**，中间随时可能让你 **怀疑人生**。
- 很多散户不是在终点卖出的，是在 **中途恐慌** 时割肉的——这就是「体验」的力量。

接下来，我们要给这种「颠」找一个 **可量化的名字**。

---

### 5.2 什么是波动？

这一节 **不急着下定义**。我们先看现象。

金融市场中，**价格从来不是直线运动的**。

- 有些股票像 **坐电梯**——缓缓上行，偶尔停一停
- 有些股票像 **坐过山车**——尖叫、俯冲、再弹射

量化里，我们把这种 **「不稳定程度」** 叫做 **波动（Volatility）**。

注意：这里说的是 **波动**，还不是完整的 **波动率指标**——那个要在后面用数据算出来。现在先用肉眼建立直觉。

**目的**：并排对比「电梯型」和「过山车型」走势，并标出 B 从高点回落的部分。

In [ ]:
# ========== 5.2 电梯 vs 过山车 ==========
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

axes[0].plot(x, stock_a, 'o-', color='#007AFF', linewidth=2.5, markersize=8)
axes[0].set_title('股票 A · 像坐电梯\n上涨平稳', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels(days, fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, stock_b, 's-', color='#E82127', linewidth=2.5, markersize=8)
peak = np.maximum.accumulate(stock_b)
axes[1].fill_between(x, stock_b, peak, alpha=0.35, color='#E82127', label='从高点回落')
axes[1].set_title('股票 B · 像坐过山车\n大起大落', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels(days, fontsize=9)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.set_ylabel('价格')

fig.suptitle('市场中的「不稳定程度」不同', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**结果怎么读？**

- 右图红色阴影 = **从阶段高点跌下来** 的部分，也就是你账户「缩水」的时刻。
- 量化研究需要一把 **尺子** 来测量这种不稳定——这把尺子，后面就叫 **波动率**。

但尺子不能直接在「价格」上量。为什么呢？进入下一节。

---
### 5.3 用收益率观察波动

专业量化研究 **很少直接比较价格**，而更常研究 **收益率**。

为什么？

| 问题 | 解释 |
|------|------|
| 价格会「越涨越高」 | 100 元的 5% 和 200 元的 5%，绝对金额差一倍 |
| 不同股票价位差很多 | 直接比价格涨跌 **不公平** |
| 收益率是 **相对变化** | 可以在不同股票、不同时期之间比较 |

**日收益率**（第一期学过，这里正式写一遍）：

$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$

👉 **[🎬 点击这里：观看日收益率动态演示](../../assets/interactive/daily-return-demo.html)**（在新页面打开；橙色 = 昨天 $P_{t-1}$，蓝色 = 今天 $P_t$，看公式如何逐日代入）


代码一行：

```python
df['return'] = df['Close'].pct_change()
```

**目的**：先通过上方链接理解公式如何一天天计算；再下载苹果（AAPL）近一年真实数据，画累计曲线与直方图。

我们要画 **两张图**：

1. **累计收益率曲线** —— 看长期方向
2. **收益率直方图** —— 看日常分布是 **集中** 还是 **分散**


In [ ]:
# ========== 5.3 收益率曲线 + 直方图 ==========
TICKER = 'AAPL'
raw = yf.download(TICKER, period='1y', progress=False)
df = pd.DataFrame({'Close': get_close(raw, TICKER)})
df['return'] = df['Close'].pct_change()
df = df.dropna().copy()

df['cum_return'] = (1 + df['return']).cumprod() - 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(df.index, df['cum_return'] * 100, color='#007AFF', linewidth=1.5)
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_title(f'{TICKER} · 累计收益率曲线', fontsize=13)
axes[0].set_ylabel('累计收益率 (%)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(df['return'] * 100, bins=30, color='#007AFF', alpha=0.75, edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title(f'{TICKER} · 日收益率直方图', fontsize=13)
axes[1].set_xlabel('日收益率 (%)')
axes[1].set_ylabel('天数')

fig.suptitle('用收益率观察波动：直方图越「散」，日常越颠', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'样本: {len(df)} 个交易日')
print(f'累计收益: {df["cum_return"].iloc[-1]:.2%}')

**结果怎么读？**

- **左图** 回答：「这一年总体赚还是亏？」
- **右图** 回答：「大多数日子的涨跌，集中在哪个区间？」
- 如果直方图 **又宽又胖**，说明大涨大跌的日子多 → 通常 **波动更大**。
- 如果直方图 **又窄又高**，说明涨跌集中在均值附近 → 通常 **更平稳**。

问题来了：**有没有一个数字，能量化这种「分散程度」？**

---

### 5.4 波动率是什么？

先用 **人话**：

- 收益率 **上蹿下跳很厉害** → 波动率 **高**
- 收益率 **变化不大、比较集中** → 波动率 **低**

怎么量化「分散程度」？统计学里有一个经典工具：**标准差（Standard Deviation）**。

直觉翻译：

| 标准差 | 含义 |
|--------|------|
| **越大** | 收益率越分散 → 波动率越高 |
| **越小** | 收益率越集中 → 波动率越低 |

在量化金融里，有一个极其常用的近似：

> **波动率 ≈ 收益率的标准差**

样本标准差公式（**不用推导**，先会用）：

$$
\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(r_i - \bar{r})^2}
$$

👉 **[🎬 点击这里：观看样本标准差动态演示](../../assets/interactive/std-dev-demo.html)**（在新页面打开；逐步显示 $r_i$、$\bar{r}$、$(r_i-\bar{r})^2$ 如何累加并算出 $\sigma$）

符号说明：

| 符号 | 含义 |
|------|------|
| $r_i$ | 第 $i$ 天的日收益率 |
| $\bar{r}$ | 日收益率的平均值 |
| $n$ | 样本天数 |
| $\sigma$ | 标准差（我们用来近似 **日波动率**） |


# 波动率度量与区间预测（期权交易应用）

在期权交易中，我们不仅需要对未来波动率做**单点估计**，还必须判断波动率大概率所处的区间。
想要实现这一目标，就需要研究**波动率锥**的构建逻辑与抽样特性。

精准度量波动率、预测波动率分布，是期权交易取得成功的关键。但这两点**并非盈利的充分条件**。
不能单纯因为波动率偏低就买入，也不能只因波动率偏高就卖出。市场中波动率定价偏高或偏低，背后都有对应的基本面逻辑。

所有波动率预测，都需要搭配基本面分析作为补充：
- 哪些因素会推动波动率上行？
- 持有波动率空头头寸时，需要警惕哪些风险事件？

金融市场结构复杂、资产之间高度联动，所有波动率测算与预测结果，都必须结合当下交易环境综合判断。

波动率的度量与价格估计存在本质区别：**瞬时波动率无法直接观测**，其真实水平需要时间验证。
同时波动率测算可选用的统计指标繁多，因此波动率度量也被视作一门“经验艺术”。
Poon (2005) 汇总了超过 100 篇波动率预测相关文献，也侧面说明了该领域的研究难度。

本文不追求给出唯一标准答案，主要介绍各类波动率估计方法，对比其优劣，并说明不同场景下的适用选择。

---

## 波动率的基础定义

波动率的标准定义为**方差的平方根**。

### 1. 总体方差公式
总体方差计算公式：
$$
s^2 = \frac{1}{N}\sum_{j=1}^N (x_j - \bar{x})^2 \tag{2-1a}
$$

👉 **[🎬 点击这里：观看总体方差动态演示](../../assets/interactive/population-variance-demo.html)**（在新页面打开；逐步显示 $x_j$、$\bar{x}$、$(x_j-\bar{x})^2$ 如何累加，并除以 **$N$** 算出 $s^2$）


tips:很多金融入门材料会不严谨把它叫样本方差，严格来说这叫总体方差：当你手里的数据就是研究的全部总体、不需要外推更大群体时，直接除以数据量
N；它如果强行当 “样本方差” 用，会系统性低估总体真实方差，属于有偏估计

参数说明：
- $x_j$：资产对数收益率
- $\bar{x}$：样本平均收益率
- $N$：样本数量

### 2. 年化波动率处理
实际应用中需要将方差转为**年化形式**，计算时需乘以年化交易天数。
行业通用标准：美股市场一年交易日为 **252 天**，使用日频数据时，年化因子取 252。

### 3. 价格序列调整：除息与拆股
如果原始价格序列包含**股息派发、股票拆股**等事件，必须提前对价格进行修正。

除息会导致股价被动下跌，即便标的本身没有真实波动，也会被计算出虚假波动，最终让波动率测算产生数个百分点的误差。

**举例**
若除息造成单日股价下跌 3%，年化后虚假波动为：
$$
0.03 \times \sqrt{252} \approx 48\%
$$
虚假波动会严重干扰判断，因此必须做价格修正。

常用两种调整方式：
1. **简单扣减法**
    在除息日，用除息前股价直接减去股息。该方式能保证除息前后股价单日绝对变动不变。
    缺点：若分红次数多、金额大，调整后可能出现**负股价**，逻辑失效。

2. **系数修正法（推荐）**
    引入百分比不变的调整因子：$\displaystyle \frac{\text{股价}}{\text{股价}-\text{股息}}$
    - 向后调整：对除息日之前所有历史股价乘以修正系数
    - 向前调整：对除息日之后股价做修正，调整后当前股价会与原价格不一致

---

## 结合收益率分布的方差修正

公式 $(2-1a)$ 仅为纯统计求和，**不依赖任何分布假设**，任意有限样本都可计算。

但在期权定价场景中，需要对收益率生成过程做出假设。经典 **BSM 模型** 假定：
**资产对数收益率服从正态分布**。

尽管该假设与真实市场存在偏差，但方差与波动率依旧是描述收益率分布离散程度的核心指标。

### 1. 忽略均值的简化方差公式
金融研究中，收益率均值（漂移项）估计难度大，小样本下误差尤其明显，还会引入额外噪声。
行业普遍做法：将平均收益率项置为 0，简化后公式：
$$
s^2 = \frac{1}{N}\sum_{j=1}^N x_j^2 \tag{2-1b}
$$

👉 **[🎬 动态演示：(2-1b) 忽略均值简化方差](../../assets/interactive/variance-formulas-demo.html#2-1b)**（新页面打开；逐步代入 $x_j^2$ 并除以 $N$）

### 2. 总体方差的无偏估计
公式 $(2-1a)$ 为样本方差，若用样本推断总体方差，需要进行自由度修正（Kenny & Keeping, 1951）：
$$
\hat{\sigma}^2 = \frac{N}{N-1} \cdot s^2 \tag{2-2}
$$

也有大量应用直接将方差定义为**总体方差无偏估计**，省去二次转换：
$$
s^2 = \frac{1}{N-1}\sum_{j=1}^N (x_j - \bar{x})^2 \tag{2-3}
$$

👉 **[🎬 动态演示：(2-2)(2-3) 无偏方差估计](../../assets/interactive/variance-formulas-demo.html#2-2)**（对比 ÷$N$ 与 ÷$N-1$，验证 $\hat{\sigma}^2 = \frac{N}{N-1}s^2$）

> 注意：
> 方差公式分母是 $N$ 还是 $N-1$，是实务中计算结果出现分歧的主要原因。
> Excel 中 `VAR` 函数默认采用公式 $(2-3)$ 的计算逻辑。

### 3. 詹森不等式与波动率偏差校正
即使得到无偏的方差估计，对方差开平方得到的**样本波动率（标准差）** 依然存在低估，该问题由**詹森不等式**解释：
$$
E(s) = E(\sqrt{s^2}) < \sqrt{E(s^2)} = \sqrt{\sigma^2} = \sigma
$$

👉 **[🎬 动态演示：詹森不等式与波动率低估](../../assets/interactive/variance-formulas-demo.html#jensen)**（蒙特卡洛展示 $E(s) < \sigma$）

含义：样本标准差的数学期望，小于总体真实标准差，因此必须对波动率结果做偏差校正。

### 4. 正态假设下的样本标准差分布
假设收益率服从正态分布，样本标准差 $s$ 的概率密度函数可表示为样本量 $N$ 的函数：
$$
f_N(s) = \frac{2}{\Gamma\left(\frac{N-1}{2}\right)} \left( \frac{N}{2\sigma^2} \right)^{\frac{N-1}{2}}
\exp\left( -\frac{Ns^2}{2\sigma^2} \right) s^{N-2} \tag{2-4}
$$

👉 **[🎬 动态演示：(2-4) 样本标准差分布 $f_N(s)$](../../assets/interactive/variance-formulas-demo.html#2-4)**（调节 $N$ 观察 PDF 如何向 $\sigma$ 集中）

参数说明：
- $s$：样本标准差
- $\sigma$：总体标准差
- $\Gamma(x)$：伽马函数，整数满足 $\Gamma(n) = (n-1)!$
- 函数形态可参考原文图 2-1

**目的**：先用 5.1 的玩具案例验证直觉，再算 AAPL 真实数据的日波动率。

In [ ]:
# ========== 5.4 标准差 ≈ 波动率 ==========
def step_returns(prices):
    return pd.Series(prices).pct_change().dropna()

r_a = step_returns(stock_a)
r_b = step_returns(stock_b)
vol_a = float(r_a.std())
vol_b = float(r_b.std())

print('── 玩具案例（5.1）──')
print(f'股票 A 日波动率(标准差): {vol_a:.2%}')
print(f'股票 B 日波动率(标准差): {vol_b:.2%}')
print(f'B 约为 A 的 {vol_b / vol_a:.0f} 倍\n')

daily_vol = float(df['return'].std())
print('── AAPL 真实数据 ──')
print(f'日波动率(标准差): {daily_vol:.4%}')
print(f'粗略直觉: 典型单日涨跌幅度大约在 ±{daily_vol:.2%} 附近')

**结果怎么读？**

- 玩具案例里，B 的标准差 **远大于** A——和我们肉眼看到的「过山车」一致。
- AAPL 的日波动率约 **1%～2%** 量级（随样本变化），意味着「平常日子」涨跌多半在这个范围内。
- **验证成功**：标准差确实能量化 5.1 里「体验不同」的直觉。



# 课堂笔记：波动率估计的样本误差与观测局限
更多样本数据确实能提升波动率估计的精准度。这套逻辑在静态平稳过程中测算波动率完全可行，但放到真实动态金融市场里，这套方案会暴露出大量缺陷。

受抽样误差影响，若选取的样本量过小，噪声干扰会让测算出的波动率严重偏离标的真实波动水平；可反过来，若选取过多历史数据，样本里又会混入大量与当下市场环境不匹配的旧行情，失去参考价值。
> **✨核心要点**：筛选适配的样本区间极具实操技巧，最优数据长度完全取决于当期市场状态。

市场上普遍采用近 30 日收盘价估算波动率的简易做法，会产生极其夸张的抽样误差：
> **⚠️关键结论**：在 95% 置信区间（2 倍标准差）下，测算结果相对真实波动率的偏差最高可达 25%。

需要严格区分**抽样误差**与**度量误差**两个概念。
物理实验中，测量设备的精度上限会天然带来度量误差；但股票价格（忽略买卖价差后）是精确客观的数值，由价格计算得出的历史波动率本身不存在度量层面的不确定性。

真正的不确定性在于：测算得到的历史波动，未必能代表标的当下、未来的真实波动中枢。可以用棒球击球类比：一名击球手 5 次击打全部命中，他当期击球命中率是确凿的 100%，但我们绝不能判定他长期击球能力就是百分百 —— 这五击大概率只是短期高光片段，拉长周期他的表现会大幅波动。

波动率同理，
> **🔥核心金句**：它属于无法直接观测的隐变量，我们只能通过历史行情近似估算。

历史波动率仅仅是真实波动能力的片面快照，就如同短期夺冠场次，无法完整还原一名运动员的真实竞技水平。

下一节，我们用 **三只真实明星股** 来场正面 PK。

---

### 5.5 苹果 VS 特斯拉 VS 英伟达

本章第一个 **高光时刻**——用真实市场数据说话。

三只股票都是科技圈顶流，但 **脾气** 差很多：

| 代码 | 公司 |
|------|------|
| AAPL | 苹果 |
| TSLA | 特斯拉 |
| NVDA | 英伟达 |

我们要算两个数：

1. **日波动率** = `returns.std()`
2. **年化波动率** = `returns.std() * np.sqrt(252)`

**为什么要年化？为什么要乘 $\sqrt{252}$？**

- 日波动率只有「一天」尺度，不同研报、不同软件 **口径不统一**
- 业界习惯换算成 **「一年尺度」**，方便横向比较
- 252 是美股大约一年的 **交易日数**
- $\sqrt{252}$ 来自波动随时间累积的数学规律（本章 **记住结论即可**，不必推导）

> 一句话：**年化波动率 = 把「日波动」翻译成人人都懂的「年尺度」。**

**目的**：下载近 2 年数据，计算三票日/年化波动率，画柱状图对比。

In [ ]:
# ========== 5.5 三票波动率 PK ==========
TICKERS = ['AAPL', 'TSLA', 'NVDA']
NAMES = {'AAPL': '苹果', 'TSLA': '特斯拉', 'NVDA': '英伟达'}
COLORS = ['#007AFF', '#E82127', '#76B900']

multi_raw = yf.download(TICKERS, period='2y', progress=False)
multi_close = multi_raw['Close']
multi_rets = multi_close.pct_change().dropna()

records = []
ann_vols = []
for t in TICKERS:
    r = multi_rets[t]
    dvol = float(r.std())
    avol = dvol * np.sqrt(TRADING_DAYS)
    aret = float((1 + r.mean()) ** TRADING_DAYS - 1)
    ann_vols.append(avol)
    records.append({
        '名称': NAMES[t],
        '日波动率': f'{dvol:.2%}',
        '年化波动率': f'{avol:.2%}',
        '年化收益(约)': f'{aret:.2%}',
    })

vol_table = pd.DataFrame(records, index=TICKERS)
print('波动率对比表：')
display(vol_table)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([NAMES[t] for t in TICKERS], [v * 100 for v in ann_vols],
              color=COLORS, edgecolor='white', linewidth=1.2)
ax.set_ylabel('年化波动率 (%)')
ax.set_title('谁最稳？谁最刺激？· 年化波动率柱状图', fontsize=14)
ax.grid(True, axis='y', alpha=0.3)
for bar, v in zip(bars, ann_vols):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{v:.1%}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

rank = sorted(zip(TICKERS, ann_vols), key=lambda x: x[1])
print(f"最稳: {NAMES[rank[0][0]]} ({rank[0][1]:.1%})")
print(f"最刺激: {NAMES[rank[-1][0]]} ({rank[-1][1]:.1%})")

**结果怎么读？**

- 通常 **特斯拉** 年化波动率最高 → 涨得猛，跌得也猛 → 「最刺激」。
- **苹果** 往往相对低 → 大公司、机构持仓多，走势相对「稳」。
- **英伟达** 介于两者之间（具体数值随样本期变化，以你跑出来的为准）。
- 注意：**高波动 ≠ 一定不好**，但它意味着你需要 **更强的心理承受力**。

数字是冷的。下一节，我们聊 **波动率怎样在真实投资中「伤人」**——这是本章高潮。

---

### 5.6 波动率为什么会让人亏钱？

假设两位投资者，**一年后账户都赚了 20%**。

| 投资者 | 年收益 | 年化波动率 | 性格画像 |
|--------|--------|------------|----------|
| **投资者 A** | 20% | 10% | 平稳型 · 像坐电梯 |
| **投资者 B** | 20% | 60% | 过山车型 · 大起大落后仍到终点 |

终点一样，路径完全不同。

关键问题：

> **如果是你，真的能坚持持有 B 吗？**

很多人在 B 的路径上会：

- **恐慌**：账户短期缩水 30%，晚上睡不着
- **情绪交易**：在低点割肉，在高点追涨
- **提前下车**：没等到终点 +20%，反而锁定亏损

所以 Yibo 想强调：

> **波动率不仅是数学指标，也是投资体验。**  
> **体验太差，会导致错误的操作——最终「理论上的收益」你根本拿不到。**

下面用 **模拟净值曲线** 展示：同样目标收益，不同波动率下的 **持有过程**。

**目的**：模拟两位投资者 1 年的净值路径（预期收益相近，波动率 10% vs 60%），对比过程差异。

In [ ]:
# ========== 5.6 同收益、不同波动：净值路径模拟 ==========
rng = np.random.default_rng(42)
n = TRADING_DAYS
target_annual = 0.20
daily_mu = (1 + target_annual) ** (1 / n) - 1   # 换算成「日均」目标收益

vol_a, vol_b = 0.10, 0.60
sigma_a = vol_a / np.sqrt(n)
sigma_b = vol_b / np.sqrt(n)

rets_a = rng.normal(daily_mu, sigma_a, n)
rets_b = rng.normal(daily_mu, sigma_b, n)

nav_a = np.cumprod(1 + rets_a)
nav_b = np.cumprod(1 + rets_b)

def max_drawdown(nav):
    peak = np.maximum.accumulate(nav)
    return float((nav / peak - 1).min())

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

axes[0].plot(nav_a, color='#007AFF', linewidth=1.5, label=f'投资者 A · 波动率 {vol_a:.0%}')
axes[0].plot(nav_b, color='#E82127', linewidth=1.5, label=f'投资者 B · 波动率 {vol_b:.0%}')
axes[0].axhline(1.20, color='gray', linestyle='--', alpha=0.6, label='目标 +20% 参考线')
axes[0].set_ylabel('净值（起点=1）')
axes[0].set_title('同样追求 ~20% 收益：持有过程天差地别', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

dd_a = nav_a / np.maximum.accumulate(nav_a) - 1
dd_b = nav_b / np.maximum.accumulate(nav_b) - 1
axes[1].fill_between(range(n), dd_a, 0, alpha=0.4, color='#007AFF', label='A 回撤')
axes[1].fill_between(range(n), dd_b, 0, alpha=0.4, color='#E82127', label='B 回撤')
axes[1].set_ylabel('回撤')
axes[1].set_xlabel('交易日')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"投资者 A  最终净值: {nav_a[-1]:.2f}  |  最大回撤: {max_drawdown(nav_a):.1%}")
print(f"投资者 B  最终净值: {nav_b[-1]:.2f}  |  最大回撤: {max_drawdown(nav_b):.1%}")
print('\n👉 想一想：B 的中途回撤，你会不会在某个低点「受不了」而卖出？')

**结果怎么读？**

- 蓝线（A）通常 **更平滑**，最大回撤更小 → 持有体验更好。
- 红线（B）可能 **长期大幅低于起点**，即使最终拉回 → 这正是 **恐慌割肉** 的高发区。
- 模拟的最终净值每次运行略有不同（随机性），但 **路径形态** 的稳定结论是：高波动 = 深回撤 + 强情绪。

巴菲特说过类似的意思：**投资最重要的不是赚得最快，而是长期活下来。**

2020 年疫情初期，很多股票 **单日波动 10% 甚至更多**——那种时候，波动率不是纸面数字，是你手机 App 上的红绿跳动。

现在，换你来选。

---

### 5.7 小实验：你会选哪一个？

规则：**未来 5 年不能卖出**。你只能选一只。

| 选项 | 预期年收益 | 年化波动率 |
|------|------------|------------|
| **A** | 40% | 50% |
| **B** | 20% | 10% |

没有标准答案。

- 选 A 的人，可能 **野心更大**，能承受剧烈波动
- 选 B 的人，可能 **更重视睡眠**，宁可少赚也要稳

量化研究员不会简单说「哪个对」，而是会先问：

> **你的风险承受能力，配得上这个波动率吗？**

下面散点图一共 **10 个点**（8 只真实股票 + 虚构 A/B），覆盖防御型到极端投机型（**不代表未来**）。

**目的**：8 只风格不同的股票 + 2 个虚构选项，同一张散点图对比（共 10 个点）。

In [ ]:
# ========== 5.7 小实验：10 个点风险-收益散点 ==========
EXP_TICKERS = ['KO', 'JPM', 'AAPL', 'MSFT', 'META', 'NVDA', 'TSLA', 'COIN']
EXP_NAMES = {
    'KO': '可口可乐', 'JPM': '摩根大通', 'AAPL': '苹果', 'MSFT': '微软',
    'META': 'Meta', 'NVDA': '英伟达', 'TSLA': '特斯拉', 'COIN': 'Coinbase',
}
EXP_COLORS = ['#8B4513', '#003366', '#007AFF', '#00A4EF', '#0668E1', '#76B900', '#E82127', '#1652F0']

def annual_stats(series):
    s = series.dropna()
    return {
        'ret': float((1 + s.mean()) ** TRADING_DAYS - 1),
        'vol': float(s.std() * np.sqrt(TRADING_DAYS)),
    }

exp_raw = yf.download(EXP_TICKERS, period='2y', progress=False)['Close']
exp_rets = exp_raw.pct_change().dropna()

fig, ax = plt.subplots(figsize=(10, 7))

for t, c in zip(EXP_TICKERS, EXP_COLORS):
    st = annual_stats(exp_rets[t])
    ax.scatter(st['vol'], st['ret'], s=130, color=c, edgecolors='white', linewidth=1.2, zorder=3)
    ax.annotate(EXP_NAMES[t], (st['vol'], st['ret']),
                xytext=(6, 5), textcoords='offset points', fontsize=9)

ax.scatter(0.50, 0.40, s=200, facecolors='none', edgecolors='#333', linewidths=2,
           linestyle='--', label='选项 A · 40% / 50%', zorder=2)
ax.scatter(0.10, 0.20, s=200, facecolors='none', edgecolors='#333', linewidths=2,
           linestyle='--', label='选项 B · 20% / 10%', zorder=2)

ax.set_xlabel('年化波动率 →  风险 / 持有体验')
ax.set_ylabel('年化收益率 →  预期回报')
ax.set_title('5 年不能卖：10 个点，你会选哪个？', fontsize=14)
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('── 10 个点一览（近 2 年历史）──')
rows = []
for t in EXP_TICKERS:
    st = annual_stats(exp_rets[t])
    rows.append({'名称': EXP_NAMES[t], '年化收益': f"{st['ret']:.1%}", '年化波动': f"{st['vol']:.1%}"})
rows.append({'名称': '选项 A（虚构）', '年化收益': '40.0%', '年化波动': '50.0%'})
rows.append({'名称': '选项 B（虚构）', '年化收益': '20.0%', '年化波动': '10.0%'})
display(pd.DataFrame(rows))

**结果怎么读？**

- 横轴越靠右 = 波动越大 = 过程越「刺激」。
- 纵轴越高 = 历史平均收益越高（**过去不代表未来**）。
- 在下方代码格 **写下你的选择和 1～2 句理由**——这是本章最重要的「思考作业」。

In [ ]:
# ========== 写下你的选择 ==========
my_choice = 'B'   # 改成 'A' 或 'B'
my_reason = '在这里写 1～2 句话：例如「5 年不能卖，我受不了 A 的回撤，宁可少赚」'

print(f'我的选择: {my_choice}')
print(f'理由: {my_reason}')

---

### 5.8 挑战任务

逐级通关，巩固本章技能：

| 等级 | 任务 | 提示 |
|------|------|------|
| **Lv.1** | 计算 **另一只** 股票的年化波动率 | 改 ticker，复用 5.5 代码 |
| **Lv.2** | 把 **MSFT（微软）** 加入三票对比 | 扩展 `TICKERS` 列表 |
| **Lv.3** | 画 **收益率分布图**（Histogram + 正态对比可选） | 复用 5.3 直方图 |
| **Lv.4** | 找出 **近 1 年波动率最高** 的一只股票 | 自选 5～10 个 ticker 批量算 |

下面给你 **Starter 代码**——每级改一点点就能跑通。

In [ ]:
# ========== 5.8 挑战 Starter 代码 ==========
# ── Lv.1：换一只股票 ──
lv1_ticker = 'AMD'
lv1_raw = yf.download(lv1_ticker, period='1y', progress=False)
lv1_ret = get_close(lv1_raw, lv1_ticker).pct_change().dropna()
lv1_ann_vol = float(lv1_ret.std() * np.sqrt(TRADING_DAYS))
print(f'[Lv.1] {lv1_ticker} 年化波动率: {lv1_ann_vol:.2%}')

# ── Lv.2：加入 MSFT ──
lv2_tickers = ['AAPL', 'TSLA', 'NVDA', 'MSFT']
lv2_raw = yf.download(lv2_tickers, period='1y', progress=False)['Close']
lv2_rets = lv2_raw.pct_change().dropna()
lv2_vols = lv2_rets.std() * np.sqrt(TRADING_DAYS)
print('\n[Lv.2] 含 MSFT 的年化波动率：')
print(lv2_vols.sort_values(ascending=False).map(lambda x: f'{float(x):.2%}'))

# ── Lv.3：收益率分布图（以 MSFT 为例）──
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(lv2_rets['MSFT'] * 100, bins=30, color='#00A4EF', alpha=0.75, edgecolor='white')
ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('[Lv.3] MSFT 日收益率分布', fontsize=13)
ax.set_xlabel('日收益率 (%)')
ax.set_ylabel('天数')
plt.tight_layout()
plt.show()

# ── Lv.4：近 1 年波动率最高的股票 ──
candidates = ['AAPL', 'TSLA', 'NVDA', 'MSFT', 'AMD', 'META', 'NFLX', 'COIN']
cand_raw = yf.download(candidates, period='1y', progress=False)['Close']
cand_vols = cand_raw.pct_change().dropna().std() * np.sqrt(TRADING_DAYS)
cand_vols = cand_vols.sort_values(ascending=False)
print('\n[Lv.4] 近 1 年年化波动率排名：')
for t, v in cand_vols.items():
    print(f'  {t}: {float(v):.2%}')
print(f'\n🏆 波动率最高: {cand_vols.index[0]} ({float(cand_vols.iloc[0]):.2%})')

**挑战结果怎么读？**

- Lv.4 里 **COIN（Coinbase）**、**TSLA** 这类标的常常名列前茅——和直觉是否一致？
- 完成后建议 **截图一张柱状图或散点图**，作为本章通关凭证。
- 欢迎把结果发到学习群，看看和同学选的是否一样。

---

## 本章总结

很多新手只关注：

> **「我能赚多少钱？」**

而量化研究员更关心：

> **「为了赚这些钱，我要承担什么风险？」**

本章，你完成了第一次 **认知转变**：

- **涨得多 ≠ 投资体验好** —— 5.1 的 A/B 故事
- **波动率** 衡量的是路径的「不稳定程度」
- 专业分析看 **收益率**，用 **标准差** 量化波动，用 **√252** 年化对比
- **高波动** 会通过情绪与行为，让你 **拿不到** 理论上的收益

---

**收益** 吸引你进入市场。  
**风险** 决定你能否留在市场。  
**波动率** 不是市场噪音——它是 **投资成本** 的一部分。

---

**下一章预告**：《夏普比率与 Beta》—— 不只看「赚多少」和「有多颠」，还要看 **性价比** 与 **和大盘的关系**。